# DAG DSL → Graphviz

Build a model with the SKaiNET DAG DSL and return it from a cell to see it rendered as inline SVG. The DOT → SVG step runs entirely on the JVM kernel via the bundled Graphviz wasm artifact — no JS, no CDN, no browser-side runtime.

The notebook integration registers a `GraphProgram.asDot()` helper that lowers the symbolic DAG to a `ComputeGraph` and serializes it with SKaiNET's `toGraphviz` exporter, so you can write `dag { … }.asDot()` instead of chaining the two steps by hand.

In [1]:
USE {
    repositories {
        mavenCentral()
    }
    dependencies {
        implementation("sk.ainet.app:kotlin-notebook:0.25.1")
    }
}

SKaiNET Kotlin Notebook v0.25.1 ready

⚠ SKaiNET SIMD path is NOT active — falling back to scalar CPU kernels. 
 jdk.incubator.vector module not loaded — start the kernel with --add-modules jdk.incubator.vector 
 IntelliJ Kotlin Notebook: Settings → Languages & Frameworks → Kotlin → Kotlin Notebook → JVM options, add --add-modules jdk.incubator.vector . 
 Run checkSimd() for the full diagnostic.

## Define a tiny conv block

A 3×3 stride-2 convolution over a 224×224 RGB input, biased, followed by `relu`. The DSL ops (`dag`, `input`, `parameter`, `constant`, `conv2d`, `relu`, `output`), `TensorSpec`, and `asDot()` are auto-imported by the integration. Dtype markers like `FP32` are `object`s used as reified type arguments; Kotlin Jupyter's wildcard imports don't reliably reach type-argument scope, so cells using them explicit-import the dtype (every existing tutorial notebook does the same).

In [2]:
import sk.ainet.lang.types.FP32

val program = dag {
    val x = input<FP32>("input", TensorSpec("input", listOf(1, 3, 224, 224), "FP32"))

    val w = parameter<FP32, Float>("weight") { shape(64, 3, 3, 3) { ones() } }
    val b = constant<FP32, Float>("bias") { shape(64) { zeros() } }

    val conv = conv2d(x, w, b, stride = 2 to 2, padding = 1 to 1)
    val activated = relu(conv)

    output(activated)
}

program.asDot()

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 8.0.5 (20230430.1635)
 -->
<!-- Pages: 1 -->
 
 
 
<!-- input_input -->
 
 input_input 
 
 input 
 
 input_input 
 
<!-- n0_conv2d -->
 
 n0_conv2d 
 
 conv2d 
 
 n0_conv2d 
 
<!-- input_input->n0_conv2d -->
 
 input_input->n0_conv2d 
 
 
 
<!-- param_weight -->
 
 param_weight 
 
 input 
 
 param_weight 
 
<!-- param_weight->n0_conv2d -->
 
 param_weight->n0_conv2d 
 
 
 
<!-- param_weight_op -->
 
 param_weight_op 
 
 kind: parameter 
 
<!-- param_weight_op->param_weight -->
 
 param_weight_op->param_weight 
 
 
 
<!-- const_bias -->
 
 const_bias 
 
 input 
 
 const_bias 
 
<!-- const_bias->n0_conv2d -->
 
 const_bias->n0_conv2d 
 
 
 
<!-- const_bias_op -->
 
 const_bias_op 
 
 kind: const 
 
<!-- const_bias_op->const_bias -->
 
 const_bias_op->const_bias 
 
 
 
<!-- n1_relu -->
 
 n1_relu 
 
 relu 
 
 n1_relu 
 
<!-- n0_conv2d->n1_relu -->
 
 n0_conv2d->n1_relu 
 
 
 
<!-- n0_conv2d_op -->
 
 n0_conv2d_op 
 
 stride: (2, 2) 
 padding: (1, 1) 
 dilation: (1, 1) 
 groups: 1 
 hasBias: true 
 
<!-- n0_conv2d_op->n0_conv2d -->
 
 n0_conv2d_op->n0_conv2d

## Inspect the underlying DOT

`asDot()` returns a `Dot(val source: String)` — handy if you want to paste the text into a separate Graphviz tool or commit it as a fixture.

In [3]:
println(program.asDot().source)

digraph {
    rankdir=LR;
    input_input [label="input | input_input", shape=record, style=filled, fillcolor=lightblue];
    param_weight [label="input | param_weight", shape=record, style=filled, fillcolor=lightblue];
    param_weight_op [label="kind: parameter", shape=box, style=dashed];
    param_weight_op -> param_weight [style=dotted];
    const_bias [label="input | const_bias", shape=record, style=filled, fillcolor=lightblue];
    const_bias_op [label="kind: const", shape=box, style=dashed];
    const_bias_op -> const_bias [style=dotted];
    n0_conv2d [label="conv2d | n0_conv2d", shape=record];
    n0_conv2d_op [label="stride: (2, 2)\npadding: (1, 1)\ndilation: (1, 1)\ngroups: 1\nhasBias: true", shape=box, style=dashed];
    n0_conv2d_op -> n0_conv2d [style=dotted];
    n1_relu [label="relu | n1_relu", shape=record];
    input_input -> n0_conv2d;
    param_weight -> n0_conv2d;
    const_bias -> n0_conv2d;
    n0_conv2d -> n1_relu;
}



## Vertical layout

`asDot(rankdir = "TB")` flips the graph from left-to-right to top-to-bottom — easier to read for deep stacks.

In [4]:
renderDot(program.asDot(rankdir = "TB")) {
    maxHeight = "720px"
}

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 8.0.5 (20230430.1635)
 -->
<!-- Pages: 1 -->
 
 
 
<!-- input_input -->
 
 input_input 
 
 input 
 
 input_input 
 
<!-- n0_conv2d -->
 
 n0_conv2d 
 
 conv2d 
 
 n0_conv2d 
 
<!-- input_input->n0_conv2d -->
 
 input_input->n0_conv2d 
 
 
 
<!-- param_weight -->
 
 param_weight 
 
 input 
 
 param_weight 
 
<!-- param_weight->n0_conv2d -->
 
 param_weight->n0_conv2d 
 
 
 
<!-- param_weight_op -->
 
 param_weight_op 
 
 kind: parameter 
 
<!-- param_weight_op->param_weight -->
 
 param_weight_op->param_weight 
 
 
 
<!-- const_bias -->
 
 const_bias 
 
 input 
 
 const_bias 
 
<!-- const_bias->n0_conv2d -->
 
 const_bias->n0_conv2d 
 
 
 
<!-- const_bias_op -->
 
 const_bias_op 
 
 kind: const 
 
<!-- const_bias_op->const_bias -->
 
 const_bias_op->const_bias 
 
 
 
<!-- n1_relu -->
 
 n1_relu 
 
 relu 
 
 n1_relu 
 
<!-- n0_conv2d->n1_relu -->
 
 n0_conv2d->n1_relu 
 
 
 
<!-- n0_conv2d_op -->
 
 n0_conv2d_op 
 
 stride: (2, 2) 
 padding: (1, 1) 
 dilation: (1, 1) 
 groups: 1 
 hasBias: true 
 
<!-- n0_conv2d_op->n0_conv2d -->
 
 n0_conv2d_op->n0_conv2d